# Aula 09 — Backward da camada afim

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joaopaulomirandamatias/ai-lab/blob/main/04-deep-learning/m5-redes-neurais-do-zero/notebooks/09-backward-camada-afim-laboratorio.ipynb)

Laboratório reproduzível em **NumPy puro** para derivar, implementar e testar o backward de

$$Z=XW+b.$$

O arquivo versionado não contém outputs. Execute as células em ordem para reproduzir os resultados.

## Objetivos

- Implementar forward e backward com contratos explícitos de rank, shape e finitude.
- Confirmar $dX=GW^\top$, $dW=X^\top G$ e $db=\sum_i G_{i,:}$.
- Confrontar a versão vetorizada com laços independentes.
- Executar gradient checking coordenado e um teste direcional da VJP.
- Demonstrar a média duplicada, a acumulação por microbatches e o risco de cache mutável.

## Ambiente

- Python >= 3.11
- NumPy >= 1.26
- Matplotlib >= 3.8
- nbformat >= 5.9 apenas para validação do arquivo

Não há download, credencial, framework de deep learning nem fonte de aleatoriedade sem seed.

In [ ]:
from importlib.metadata import version
import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

SEED = 20260909
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=8, suppress=True)

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("nbformat:", version("nbformat"))
print("Seed:", SEED)

## 1. Implementação estrita

Usaremos exemplos nas linhas. Para $X\in\mathbb{R}^{m\times d_{in}}$, $W\in\mathbb{R}^{d_{in}\times d_{out}}$ e $b\in\mathbb{R}^{d_{out}}$, o upstream $G$ deve ter o mesmo shape de $Z$.

O cache guarda cópias para que o backward corresponda ao forward original, mesmo se o código chamador modificar os arrays depois.

In [ ]:
def _finite_float64(name, value, rank):
    arr = np.asarray(value, dtype=np.float64)
    if arr.ndim != rank:
        raise ValueError(f"{name} deve ter rank {rank}; recebido {arr.shape}")
    if not np.all(np.isfinite(arr)):
        raise ValueError(f"{name} contém valor não finito")
    return arr


def affine_forward(X, W, b):
    X = _finite_float64("X", X, 2)
    W = _finite_float64("W", W, 2)
    b = _finite_float64("b", b, 1)
    if X.shape[1] != W.shape[0]:
        raise ValueError(f"d_in incompatível: X {X.shape} e W {W.shape}")
    if b.shape[0] != W.shape[1]:
        raise ValueError(f"d_out incompatível: W {W.shape} e b {b.shape}")
    Z = X @ W + b
    assert Z.shape == (X.shape[0], W.shape[1])
    return Z, (X.copy(), W.copy())


def affine_backward(G, cache):
    X, W = cache
    G = _finite_float64("G", G, 2)
    expected = (X.shape[0], W.shape[1])
    if G.shape != expected:
        raise ValueError(f"G deve ter shape {expected}; recebido {G.shape}")
    dX = G @ W.T
    dW = X.T @ G
    db = G.sum(axis=0)
    assert dX.shape == X.shape
    assert dW.shape == W.shape
    assert db.shape == (W.shape[1],)
    return dX, dW, db

## 2. Exemplo resolvido

As dimensões $m=2$, $d_{in}=3$ e $d_{out}=2$ foram escolhidas de propósito: shapes não quadrados expõem transposições incorretas.

In [ ]:
X_small = np.array([[1.0, -2.0, 0.5], [0.0, 3.0, -1.0]])
W_small = np.array([[2.0, -1.0], [0.5, 4.0], [-3.0, 2.0]])
b_small = np.array([0.25, -0.75])
G_small = np.array([[1.0, -2.0], [0.5, 3.0]])

Z_small, cache_small = affine_forward(X_small, W_small, b_small)
dX_small, dW_small, db_small = affine_backward(G_small, cache_small)

expected_Z = np.array([[-0.25, -8.75], [4.75, 9.25]])
expected_dX = np.array([[4.0, -7.5, -7.0], [-2.0, 12.25, 4.5]])
expected_dW = np.array([[1.0, -2.0], [-0.5, 13.0], [0.0, -4.0]])
expected_db = np.array([1.5, 1.0])

assert np.array_equal(Z_small, expected_Z)
assert np.array_equal(dX_small, expected_dX)
assert np.array_equal(dW_small, expected_dW)
assert np.array_equal(db_small, expected_db)

print("Z =", Z_small)
print("dX =", dX_small)
print("dW =", dW_small)
print("db =", db_small)

### Ledger de shapes

Uma função pequena transforma o contrato em saída auditável. Cada gradiente deve recuperar o shape do argumento em relação ao qual deriva.

In [ ]:
ledger = {
    "X": X_small.shape,
    "W": W_small.shape,
    "b": b_small.shape,
    "Z": Z_small.shape,
    "G": G_small.shape,
    "dX": dX_small.shape,
    "dW": dW_small.shape,
    "db": db_small.shape,
}
assert ledger["X"] == ledger["dX"]
assert ledger["W"] == ledger["dW"]
assert ledger["b"] == ledger["db"]
for name, shape in ledger.items():
    print(f"{name:>2}: {shape}")

## 3. Contraprova independente com laços

A versão por laços implementa diretamente as somas por índices. Ela é lenta, mas reduz o risco de confirmar uma fórmula vetorizada com outra expressão algébrica idêntica.

In [ ]:
def affine_backward_loops(G, X, W):
    m, d_in = X.shape
    d_out = W.shape[1]
    dX = np.zeros_like(X)
    dW = np.zeros_like(W)
    db = np.zeros(d_out, dtype=np.float64)
    for i in range(m):
        for j in range(d_in):
            for k in range(d_out):
                dX[i, j] += G[i, k] * W[j, k]
                dW[j, k] += X[i, j] * G[i, k]
        for k in range(d_out):
            db[k] += G[i, k]
    return dX, dW, db


loop_grads = affine_backward_loops(G_small, X_small, W_small)
vec_grads = (dX_small, dW_small, db_small)
loop_errors = [float(np.max(np.abs(a - b))) for a, b in zip(loop_grads, vec_grads)]
assert max(loop_errors) == 0.0
print("Erros máximos laços × vetorizado:", loop_errors)

## 4. Gradient checking coordenada a coordenada

Para manter o upstream constante, definimos a loss escalar

$$L(X,W,b)=\langle XW+b,G\rangle_F.$$

Suas derivadas analíticas são exatamente as três fórmulas da camada afim. A diferença central usa $h=10^{-6}$ e `float64`.

In [ ]:
def scalar_loss(X, W, b, G):
    Z, _ = affine_forward(X, W, b)
    return float(np.sum(Z * G))


def central_gradient(argument, evaluate, h=1e-6):
    argument = np.asarray(argument, dtype=np.float64)
    result = np.empty_like(argument)
    for index in np.ndindex(argument.shape):
        plus = argument.copy()
        minus = argument.copy()
        plus[index] += h
        minus[index] -= h
        result[index] = (evaluate(plus) - evaluate(minus)) / (2.0 * h)
    return result


m, d_in, d_out = 5, 4, 3
X = rng.normal(size=(m, d_in))
W = rng.normal(size=(d_in, d_out))
b = rng.normal(size=d_out)
G = rng.normal(size=(m, d_out))
_, cache = affine_forward(X, W, b)
dX, dW, db = affine_backward(G, cache)

num_dX = central_gradient(X, lambda value: scalar_loss(value, W, b, G))
num_dW = central_gradient(W, lambda value: scalar_loss(X, value, b, G))
num_db = central_gradient(b, lambda value: scalar_loss(X, W, value, G))

def max_symmetric_relative(a, b):
    scale = np.maximum(1.0, np.maximum(np.abs(a), np.abs(b)))
    return float(np.max(np.abs(a - b) / scale))


coordinate_errors = {
    "X": max_symmetric_relative(dX, num_dX),
    "W": max_symmetric_relative(dW, num_dW),
    "b": max_symmetric_relative(db, num_db),
}
assert max(coordinate_errors.values()) < 2e-9
for name, error in coordinate_errors.items():
    print(f"Erro relativo máximo em {name}: {error:.3e}")

## 5. Teste direcional da VJP

Em vez de perturbar uma coordenada por vez, perturbamos $X$, $W$ e $b$ simultaneamente nas direções $\Delta X$, $\Delta W$ e $\Delta b$. O produto interno com os gradientes analíticos deve coincidir com a derivada direcional numérica.

In [ ]:
delta_X = rng.normal(size=X.shape)
delta_W = rng.normal(size=W.shape)
delta_b = rng.normal(size=b.shape)
h = 1e-6

analytic_directional = float(
    np.sum(dX * delta_X) + np.sum(dW * delta_W) + np.sum(db * delta_b)
)
plus = scalar_loss(X + h * delta_X, W + h * delta_W, b + h * delta_b, G)
minus = scalar_loss(X - h * delta_X, W - h * delta_W, b - h * delta_b, G)
numeric_directional = (plus - minus) / (2.0 * h)
directional_error = abs(analytic_directional - numeric_directional) / max(
    1.0, abs(analytic_directional), abs(numeric_directional)
)

assert directional_error < 2e-9
print(f"Derivada direcional analítica: {analytic_directional:.12f}")
print(f"Derivada direcional numérica:  {numeric_directional:.12f}")
print(f"Erro relativo: {directional_error:.3e}")

## 6. Por que `db` usa soma, mesmo com loss média

Suponha que `raw_G` contenha gradientes por exemplo e que a loss final seja a média. Então `G_mean = raw_G / m` já carrega o fator $1/m$. Aplicar outra média em `axis=0` divide o resultado novamente.

In [ ]:
m_demo, d_out_demo = 8, 3
raw_G = rng.normal(size=(m_demo, d_out_demo))
G_mean = raw_G / m_demo

db_correct = G_mean.sum(axis=0)
db_double_mean = G_mean.mean(axis=0)
ratio = np.divide(
    db_correct,
    db_double_mean,
    out=np.full_like(db_correct, np.nan),
    where=np.abs(db_double_mean) > 1e-15,
)

assert np.allclose(db_correct, raw_G.mean(axis=0))
assert np.allclose(db_double_mean * m_demo, db_correct)
assert np.allclose(ratio, m_demo)
print("db correto:       ", db_correct)
print("db com média dupla:", db_double_mean)
print("Fator de redução indevida:", ratio)

## 7. Acumulação por microbatches

Para uma loss somada, os gradientes de $W$ e $b$ são aditivos entre partições do lote; $dX$ é a concatenação das parcelas. Isso fundamenta a acumulação de gradientes. Para uma loss média global, a ponderação deve respeitar o número total de exemplos.

In [ ]:
m_acc, d_in_acc, d_out_acc = 11, 5, 4
X_acc = rng.normal(size=(m_acc, d_in_acc))
W_acc = rng.normal(size=(d_in_acc, d_out_acc))
b_acc = rng.normal(size=d_out_acc)
G_acc = rng.normal(size=(m_acc, d_out_acc))

_, full_cache = affine_forward(X_acc, W_acc, b_acc)
full_dX, full_dW, full_db = affine_backward(G_acc, full_cache)

cut = 3
parts = []
for slc in (slice(0, cut), slice(cut, None)):
    _, part_cache = affine_forward(X_acc[slc], W_acc, b_acc)
    parts.append(affine_backward(G_acc[slc], part_cache))

micro_dX = np.vstack([parts[0][0], parts[1][0]])
micro_dW = parts[0][1] + parts[1][1]
micro_db = parts[0][2] + parts[1][2]
micro_errors = {
    "dX": float(np.max(np.abs(full_dX - micro_dX))),
    "dW": float(np.max(np.abs(full_dW - micro_dW))),
    "db": float(np.max(np.abs(full_db - micro_db))),
}
assert max(micro_errors.values()) < 2e-15
print("Erros lote completo × microbatches:", micro_errors)

## 8. Cache e mutação

O backward de $W$ depende de $X$; o backward de $X$ depende de $W$. Se o cache mantiver apenas referências e esses arrays forem alterados, o gradiente calculado já não corresponde ao forward original.

In [ ]:
X_mut = rng.normal(size=(4, 3))
W_mut = rng.normal(size=(3, 2))
b_mut = rng.normal(size=2)
G_mut = rng.normal(size=(4, 2))

_, safe_cache = affine_forward(X_mut, W_mut, b_mut)
reference_grads = affine_backward(G_mut, safe_cache)

unsafe_cache = (X_mut, W_mut)
X_mut[:] = 999.0
W_mut[:] = -999.0
safe_after_mutation = affine_backward(G_mut, safe_cache)
unsafe_after_mutation = affine_backward(G_mut, unsafe_cache)

safe_error = max(
    float(np.max(np.abs(a - b))) for a, b in zip(reference_grads, safe_after_mutation)
)
unsafe_error = max(
    float(np.max(np.abs(a - b))) for a, b in zip(reference_grads, unsafe_after_mutation)
)
assert safe_error == 0.0
assert unsafe_error > 100.0
print(f"Erro com cache defensivo: {safe_error:.1f}")
print(f"Erro com cache mutável:   {unsafe_error:.3f}")

## 9. Contratos rejeitam ambiguidade 1D

Manter um lote unitário como `(1, d_in)` impede que o eixo dos exemplos desapareça. A célula captura a exceção esperada e confirma que o contrato falha cedo.

In [ ]:
try:
    affine_forward(np.ones(3), np.ones((3, 2)), np.zeros(2))
except ValueError as exc:
    message_1d = str(exc)
else:
    raise AssertionError("X unidimensional deveria ter sido rejeitado")

assert "rank 2" in message_1d
print("Contrato acionado:", message_1d)

## 10. Magnitudes dos gradientes

O gráfico mostra a média do valor absoluto e a norma L2 de cada grupo de parâmetros no experimento de gradient checking. As escalas não são diretamente comparáveis quando os grupos têm quantidades diferentes de elementos; a figura serve para inspeção, não para declarar que um tensor é “mais importante”.

In [ ]:
names = ["dX", "dW", "db"]
gradients = [dX, dW, db]
mean_abs = [float(np.mean(np.abs(value))) for value in gradients]
l2_norm = [float(np.linalg.norm(value)) for value in gradients]

positions = np.arange(len(names))
width = 0.36
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(positions - width / 2, mean_abs, width, label="média |gradiente|")
ax.bar(positions + width / 2, l2_norm, width, label="norma L2")
ax.set_xticks(positions, names)
ax.set_ylabel("Magnitude")
ax.set_title("Diagnóstico de magnitude por grupo")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

assert all(np.isfinite(mean_abs))
assert all(np.isfinite(l2_norm))
print("Texto alternativo: barras com média absoluta e norma L2 de dX, dW e db.")
print("Média absoluta:", dict(zip(names, mean_abs)))
print("Norma L2:", dict(zip(names, l2_norm)))

## 11. Auditoria final

Os contratos abaixo consolidam as evidências do laboratório. Eles verificam resultados manuais, shapes, implementação independente, gradientes numéricos, VJP direcional, semântica da redução, microbatches, cache e finitude.

In [ ]:
checks = {
    "forward manual": np.array_equal(Z_small, expected_Z),
    "dX manual": np.array_equal(dX_small, expected_dX),
    "dW manual": np.array_equal(dW_small, expected_dW),
    "db manual": np.array_equal(db_small, expected_db),
    "shapes recuperados": all(
        (a.shape == b.shape) for a, b in ((dX, X), (dW, W), (db, b))
    ),
    "laços independentes": max(loop_errors) == 0.0,
    "gradient checking": max(coordinate_errors.values()) < 2e-9,
    "VJP direcional": directional_error < 2e-9,
    "média não duplicada": np.allclose(db_correct, raw_G.mean(axis=0)),
    "microbatches aditivos": max(micro_errors.values()) < 2e-15,
    "cache defensivo": safe_error == 0.0,
    "mutação detectável": unsafe_error > 100.0,
    "contrato 1D": "rank 2" in message_1d,
    "valores finitos": all(np.all(np.isfinite(value)) for value in gradients),
}
assert all(checks.values())
for name, passed in checks.items():
    print(f"[{'OK' if passed else 'FALHOU'}] {name}")
print(f"{len(checks)} grupos de verificações concluídos.")

## Conclusões

- A implementação vetorizada reproduziu exatamente o exemplo manual e a versão por laços.
- Diferenças centrais validaram $dX$, $dW$ e $db$; o teste direcional validou a VJP conjunta.
- Usar média em `db` depois de um upstream já normalizado reduziu indevidamente o gradiente pelo tamanho do lote.
- Gradientes de parâmetros para uma loss somada foram aditivos entre microbatches.
- Cópias no cache didático preservaram a correspondência com o forward diante de mutações externas.

Na próxima aula, o upstream atravessará sigmoid, tanh, ReLU e variantes por derivadas elemento a elemento, máscaras e análise de saturação.